<!-- # Pipeline smoke test

Raw Citi Bike data -> `RawModelData` -> `ResolvedModelData` -> `Environment` -> `SimulationLog`.

The base scenario reproduces historical **demand** exactly. `FormDeparturesPhase`
re-releases every historical departure (gated by stock), and
`FormPotentialTripsPhase` assigns each departure a destination and duration from
the OD demand model `P(target | source, commodity)` + mean historical duration.

To make the replay exact, the resolved data is built with `saturate_stock=True`:
artificial saturated stock and dock capacities replace the GBFS snapshot, so the
demand gate and the overflow redirect stay in the pipeline but never bind. (The
GBFS snapshot is a current observation, unrelated to the historical start state,
and would starve the replay with stockouts that never happened.)

Because targets and durations come from the (aggregate) OD model rather than each
trip's own record, the per-trip journal is **not** identical to history, and the
OD-count matrix drifts slightly under per-period largest-remainder rounding. The
marginal that is preserved exactly is the departures table:
`simulated_departures_df == historical_departures_df`. -->

In [1]:
import pandas as pd

from gbp.loaders.dataloader_raw import RawModelData
from gbp.loaders.dataloader_graph import ResolvedModelData, attach_simulation
from gbp.consumers.simulator.engine import Environment, EnvironmentConfig
from gbp.consumers.simulator import (
    DockArrivals,
    FormDeparturesPhase,
    FormPotentialTripsPhase,
)

In [13]:
# Raw model data: read the trip CSV and the live GBFS feed, derive raw entities.
raw_data = RawModelData(
    gbfs_base="https://gbfs.citibikenyc.com/gbfs/en",
    trips_path="/Users/vladislav/Documents/vlzm/GFDRR/data/raw/202601-citibike-tripdata_1.csv",
    seed=42,
    n_depots=10,
    depot_capacity=9000,
    n_trucks=5,
    truck_capacity_bikes=20,
    truck_rate=50.0,
    electric_bike_rate=5,
    classic_bike_rate=3,
)

# Graph (resolved) model data: period grid, historical flows, replay demand.
# saturate_stock=True swaps the GBFS snapshot for artificial saturated stock and
# capacities, so demand gating and overflow redirect stay in the pipeline but
# never bind -- the base replay reproduces the historical departures exactly.
graph_data = ResolvedModelData(raw_data, period_len=pd.Timedelta(hours=1), scale_capacity_factor = 10)

historical_flows_df = graph_data.historical_flows_df

In [26]:
graph_data.initial_inventory_df['quantity'] = 50
graph_data.facilities_capacities_df['capacity'] = 1000

In [27]:
phases_canonical = [
    DockArrivals("previous"),
    FormDeparturesPhase(),
    FormPotentialTripsPhase(),
    DockArrivals("same"),
]

env_canonical = Environment(
    graph_data,
    EnvironmentConfig(phases=phases_canonical, 
                      seed=42, 
                      scenario_id="historical_replay", 
                      demand_scale_factor=1.0,
                      number_of_periods = 30),
)
env_canonical.run()

# Wire the finished run back into the graph-data container's simulated_* slots.
attach_simulation(graph_data, env_canonical.simulated_flows_df)
simulated_flows_df = graph_data.simulated_flows_df
simulated_departures_df = graph_data.simulated_departures_df
historical_departures_df = graph_data.historical_departures_df


def _sorted(df):
    return (df.sort_values(["period_id", "facility_id", "commodity_category"])
            .reset_index(drop=True))


# The base scenario reproduces historical demand exactly: the departures marginal
# of the simulated journal equals the historical one. (The per-trip journal is not
# identical, because targets/durations are drawn from the aggregate OD model.)
# pd.testing.assert_frame_equal(_sorted(simulated_departures_df), _sorted(historical_departures_df[historical_departures_df['period_id'].isin(simulated_flows_df['period_id'].unique())]))
# print("simulated_departures_df == historical_departures_df:",
#       _sorted(simulated_departures_df).equals(_sorted(_sorted(historical_departures_df[historical_departures_df['period_id'].isin(simulated_flows_df['period_id'].unique())]))))

In [28]:
simulated_flows_df['reason'].value_counts()

reason
dock_full    662
Name: count, dtype: Int64

In [29]:
simulated_flows_df

,event_id,period_id,flow_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
0,0,0,sim_0_0,user_trip,departed,classic_bike,6626.01,5703.13,<NA>,0,17,<NA>,<NA>,1,<NA>
1,1,4,sim_4_0,user_trip,departed,classic_bike,6224.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
2,2,4,sim_4_1,user_trip,departed,classic_bike,6257.06,6339.06,<NA>,4,28,<NA>,<NA>,1,<NA>
3,3,4,sim_4_2,user_trip,departed,classic_bike,7599.09,7599.02,<NA>,4,29,<NA>,<NA>,1,<NA>
4,4,5,sim_5_0,user_trip,departed,classic_bike,6030.04,6339.06,<NA>,5,28,<NA>,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30217,30217,29,sim_29_998,user_trip,departed,electric_bike,6025.08,6932.14,<NA>,29,29,<NA>,<NA>,1,<NA>
30218,30218,29,sim_29_998,user_trip,arrived,electric_bike,6025.08,6932.14,6932.14,29,29,29,<NA>,1,<NA>
30219,30219,29,sim_29_999,user_trip,departed,classic_bike,6030.04,6115.09,<NA>,29,29,<NA>,<NA>,1,<NA>
30220,30220,29,sim_29_999,user_trip,redirected,classic_bike,6030.04,6115.09,6233.05,29,29,29,<NA>,1,dock_full


In [32]:
graph_data.historical_od_matrix_df[(graph_data.historical_od_matrix_df['source_id'] == '6030.04')&(graph_data.historical_od_matrix_df['period_id'] == 29)].sort_values('probability')

,source_id,planned_target_id,period_id,commodity_category,count,duration,probability
499723,6030.04,5779.08,29,electric_bike,1,0,1.0
500258,6030.04,6115.09,29,classic_bike,1,0,1.0


In [37]:
graph_data.simulated_inventory_df[(graph_data.simulated_inventory_df['facility_id'] == '6115.09')&(graph_data.simulated_inventory_df['period_id'] == 29)]

,period_id,facility_id,commodity_category,quantity
66089,29,6115.09,classic_bike,47
66119,29,6115.09,electric_bike,42


In [ ]:
temp = raw_data.stations_df[['station_id']]
temp['capacity'] = 100

,station_id,lat,lng
0,6602.05,40.757570,-73.990985
1,5311.08,40.714190,-73.996730
2,6789.08,40.761940,-73.925130
3,6605.08,40.756933,-73.926223
4,5584.04,40.724537,-73.981854
...,...,...,...
2245,HB608,40.739153,-74.033082
2246,JC013,40.714584,-74.042817
2247,JC014,40.718355,-74.038914
2248,HB103,40.736982,-74.027781


In [48]:
raw_data.stations_capacities_df

,station_id,capacity
0,6053.01,22
1,4715.01,27
2,7976.08,31
3,8277.03,19
4,5584.05,69
...,...,...
2159,HB106,40
2160,HB608,15
2161,HB201,28
2162,HB603,21


In [ ]:
graph_data

In [43]:
raw_data.stations_capacities_df[raw_data.stations_capacities_df['station_id'] == '6115.09']

,station_id,capacity


In [ ]:
raw_data.trips_raw_df[['']]

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,85744AF35D7F2DF5,electric_bike,2026-01-02 05:36:24.539,2026-01-02 05:42:21.153,W 42 St & 8 Ave,6602.05,E 58 St & Madison Ave,6839.04,40.757570,-73.990985,40.763026,-73.972095,member
1,9D18958E5788880B,electric_bike,2026-01-02 15:15:11.915,2026-01-02 15:18:40.462,Division St & Bowery,5311.08,Clinton St & Grand St,5303.06_,40.714190,-73.996730,40.715738,-73.986990,member
2,B050891B7B009EE5,electric_bike,2026-01-12 10:12:51.453,2026-01-12 10:16:55.731,Broadway & 31 St,6789.08,35 Ave & 37 St,6563.12,40.761940,-73.925130,40.755733,-73.923661,member
3,0B6D7938C4EF1668,electric_bike,2026-01-01 01:03:29.712,2026-01-01 01:05:37.341,34 St & 35 Ave,6605.08,35 St & Broadway,6750.16,40.756933,-73.926223,40.760339,-73.922243,member
4,95415F60C7120CC3,electric_bike,2026-01-03 19:55:51.636,2026-01-03 20:14:57.964,E 6 St & Ave B,5584.04,W 55 St & 6 Ave,6809.09,40.724537,-73.981854,40.763189,-73.978434,member
...,...,...,...,...,...,...,...,...,...,...,...,...,...
996383,E771EB98769C54D0,electric_bike,2026-01-02 12:42:51.450,2026-01-02 12:48:28.497,Carlton Ave & Dean St,4199.12,Berkeley Pl & 6 Ave,4134.06,40.680974,-73.971010,40.676530,-73.978469,member
996384,5E1BC2953B476C78,electric_bike,2026-01-06 15:55:23.634,2026-01-06 16:00:08.435,West Thames St,5114.06,Vesey St & Church St,5216.06,40.708347,-74.017134,40.712220,-74.010472,member
996385,CC254688E4151B73,electric_bike,2026-01-05 22:21:08.536,2026-01-05 22:24:19.911,Broadway & Morris St,5033.01,Vesey St & Church St,5216.06,40.705945,-74.013219,40.712220,-74.010472,member
996386,63160D4898957580,electric_bike,2026-01-10 11:38:56.010,2026-01-10 11:52:43.521,E 85 St & York Ave,7146.04,7 Ave & Central Park South,6912.01,40.775369,-73.948034,40.766741,-73.979069,member


In [31]:
def flows_to_od_matrix(flows: pd.DataFrame) -> pd.DataFrame:
    dep = flows[flows["event_type"] == "departed"].copy()
    dep["duration"] = dep["planned_end_period"] - dep["start_period"]
    od = (
        dep.groupby(["source_id", "planned_target_id", "period_id", "commodity_category"], as_index=False)
        .agg(count=("quantity", "sum"), duration=("duration", "mean"))
    )
    totals = od.groupby(["source_id", "period_id","commodity_category"])["count"].transform("sum")
    od["probability"] = od["count"] / totals
    od["duration"] = od["duration"].round().astype("Int64")
    return od

temp = flows_to_od_matrix(graph_data.historical_flows_df)
temp[(temp['source_id'] == '6626.01')&(temp['period_id'] == 0)].sort_values('probability')

,source_id,planned_target_id,period_id,commodity_category,count,duration,probability
694278,6626.01,5703.13,0,classic_bike,1,17,1.0


In [19]:
graph_data.historical_demand_df

,period_id,facility_id,commodity_category,quantity
0,0,6626.01,classic_bike,1
1,4,6224.06,classic_bike,1
2,4,6257.06,classic_bike,1
3,4,7599.09,classic_bike,1
4,5,6030.04,classic_bike,1
...,...,...,...,...
403201,345,8782.01,classic_bike,1
403202,345,8782.01,electric_bike,1
403203,345,8795.01,electric_bike,1
403204,345,8879.02,electric_bike,1


In [20]:
init_inventory = graph_data.historical_demand_df.groupby(['facility_id','commodity_category'])['quantity'].max().reset_index().sort_values('quantity', ascending=False).reset_index(drop=True)
init_inventory['quantity'] = init_inventory['quantity'] + 10
init_inventory

,facility_id,commodity_category,quantity
0,6492.08,electric_bike,79
1,5470.10,electric_bike,73
2,5470.12,electric_bike,69
3,6584.12,electric_bike,67
4,6230.04,electric_bike,64
...,...,...,...
4360,7613.04,classic_bike,11
4361,7613.04,electric_bike,11
4362,7626.02,classic_bike,11
4363,7639.06,classic_bike,11


In [18]:
graph_data.initial_inventory_df

,facility_id,commodity_category,quantity
0,3460.01,classic_bike,100
1,2312.02,classic_bike,100
2,2472.02,classic_bike,100
3,5656.03,classic_bike,100
4,5114.06,classic_bike,100
...,...,...,...
4323,JC014,electric_bike,100
4324,HB106,electric_bike,100
4325,HB603,electric_bike,100
4326,HB201,electric_bike,100


In [15]:
from gbp.loaders.dataloader_graph import flows_to_departures
historical_departures_df = flows_to_departures(graph_data.historical_flows_df)
initial_inventory_df = historical_departures_df.groupby(['facility_id','commodity_category'])['quantity'].max().reset_index().sort_values('quantity', ascending=False).reset_index(drop=True)
initial_inventory_df['quantity'] = initial_inventory_df['quantity'] + 10
initial_inventory_df[initial_inventory_df['facility_id'] == '6948.1']

,facility_id,commodity_category,quantity


In [20]:
graph_data.historical_flows_df[graph_data.historical_flows_df['realized_target_id'] == '6948.1']

,event_id,period_id,flow_id,flow_type,event_type,commodity_category,source_id,planned_target_id,realized_target_id,start_period,planned_end_period,realized_end_period,resource_id,quantity,reason
79652,79652,51,hist_353083,user_trip,arrived,electric_bike,7123.04,6948.1,6948.1,51,51,51,<NA>,1,<NA>
79769,79769,51,hist_369912,user_trip,arrived,classic_bike,7141.07,6948.1,6948.1,50,51,51,<NA>,1,<NA>
79800,79800,51,hist_376100,user_trip,arrived,classic_bike,6986.07,6948.1,6948.1,51,51,51,<NA>,1,<NA>
85626,85626,52,hist_355559,user_trip,arrived,electric_bike,6986.07,6948.1,6948.1,51,52,52,<NA>,1,<NA>
85628,85628,52,hist_355623,user_trip,arrived,electric_bike,6676.02,6948.1,6948.1,52,52,52,<NA>,1,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1978810,1978810,345,hist_369243,user_trip,arrived,electric_bike,6847.02,6948.1,6948.1,345,345,345,<NA>,1,<NA>
1978878,1978878,345,hist_372964,user_trip,arrived,electric_bike,6498.10,6948.1,6948.1,345,345,345,<NA>,1,<NA>
1978906,1978906,345,hist_374516,user_trip,arrived,classic_bike,6816.07,6948.1,6948.1,345,345,345,<NA>,1,<NA>
1991770,1991770,346,hist_356547,user_trip,arrived,electric_bike,6432.11,6948.1,6948.1,345,346,346,<NA>,1,<NA>


In [10]:
cond_1 = graph_data.historical_inventory_df['period_id'] == 0
cond_2 = graph_data.historical_inventory_df['facility_id'] == '6948.1'

graph_data.historical_inventory_df[cond_1 & cond_2]

,period_id,facility_id,commodity_category,quantity
1115640,0,6948.1,classic_bike,0
1116000,0,6948.1,electric_bike,0


In [11]:
cond_1 = graph_data.simulated_inventory_df['period_id'] == 0
cond_2 = graph_data.simulated_inventory_df['facility_id'] == '6948.1'

graph_data.simulated_inventory_df[cond_1 & cond_2]

,period_id,facility_id,commodity_category,quantity


In [ ]:
# Run invariants I1-I4 (Tier-2 of the loss-logging design). validate_run is a
# pure check over the finished run: I1 demand split, I2 spine closure, I3 the
# live inventory equals the journal projection, I4 conservation. All are dormant
# in this saturated replay (no stockout, dock-full or redirect fires) and only
# bite above the baseline -- the same posture as the rest of the constraint logic.
from gbp.consumers.simulator.validation import validate_run

violations = validate_run(env_canonical.state, graph_data)

# Loss logging made the losses a filter on the journal. Under saturation both
# reasons are zero; the line proves the channel exists and the replay is loss-free.
lost = simulated_flows_df[simulated_flows_df["event_type"] == "lost"]
print("losses by reason:", lost.groupby("reason")["quantity"].sum().to_dict() or "none")
print("still in transit at horizon:", len(env_canonical.state.in_transit))

assert not violations, "run invariants violated:\n" + "\n".join(violations)
print("invariants I1-I4: OK")

In [ ]:
env_canonical

In [1]:

import pickle
dbg = pickle.load(open('D:\\Documents\\vlzm\\GFDRR\\temp\\dbg.pkl', 'rb'))

In [ ]:
import pickle
from gbp.consumers.simulator.mechanics import free_docks, dock_up_to_capacity, plan_overflow_redirect
from gbp.model import arrived_events
from gbp.consumers.simulator.state import adjust_inventory, dock_deltas

d = pickle.load(open('D:\\Documents\\vlzm\\GFDRR\\temp\\dbg.pkl', 'rb'))
state, resolved, period = d["state"], d["resolved"], d["period"]

t = period.period_id
it = state.in_transit
due = it[(it["planned_end_period"] == t) & (it["start_period"] < t)]

inventory  = state.state_inventory_df
capacities = resolved.facilities_capacities_df
occupied = inventory.groupby("facility_id")["quantity"].sum()
capacity = capacities.set_index("facility_id")["capacity"]
idx = capacity.index.union(occupied.index)
free = capacity.reindex(idx).fillna(0) - occupied.reindex(idx).fillna(0)
free = free.clip(lower=0).astype("int64")

if due.empty:
    print("no arrivals due, skipping docking and overflow redirect")
    docked, overflow = due, due
else:
    print(f"{len(due)} arrivals due, {free.sum()} free docks total")
    rank = due.groupby("planned_target_id").cumcount()
    capacity_here = due["planned_target_id"].map(free).fillna(0)
    fits = rank < capacity_here

    docked, overflow = due[fits], due[~fits]

In [5]:
capacities

,facility_id,capacity
0,4404.10,3000000
1,5666.11,3000000
2,5476.03,3000000
3,5238.05,3000000
4,6182.02,3000000
...,...,...
2280,depot_6,3000000
2281,depot_7,3000000
2282,depot_8,3000000
2283,depot_9,3000000


In [4]:
free

facility_id
1234.56    1000000
1964.01    1000000
2009.04    1000000
2042.01    1000000
2086.07    1000000
            ...   
depot_5    3000000
depot_6    3000000
depot_7    3000000
depot_8    3000000
depot_9    3000000
Length: 2285, dtype: int64

In [19]:
resolved.facilities_df

,facility_id,facility_category
0,4404.10,station
1,5666.11,station
2,5476.03,station
3,5238.05,station
4,6182.02,station
...,...,...
2280,depot_6,depot
2281,depot_7,depot
2282,depot_8,depot
2283,depot_9,depot
